In [14]:
import pandas as pd
import numpy as np
import os

# Configura las opciones de visualización para mantener formato numérico limpio
pd.options.display.float_format = '{:,.2f}'.format

# Define las rutas de trabajo
ruta_base = r"C:\Users\patri\Downloads"
ruta_entrada = os.path.join(ruta_base, "supermarket.csv")
ruta_salida = os.path.join(ruta_base, "supermarket_limpio.csv")

# Carga el archivo crítico asegurando compatibilidad de caracteres
df = pd.read_csv(ruta_entrada, encoding='utf-8')
df

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.14,548.97,1/5/2019,13:08,Ewallet,522.83,4.76,26.14,9.10
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.82,80.22,3/8/2019,10:29,Cash,76.40,4.76,3.82,9.60
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.22,340.53,3/3/2019,13:23,Credit card,324.31,4.76,16.22,7.40
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.29,489.05,1/27/2019,20:33,Ewallet,465.76,4.76,23.29,8.40
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.21,634.38,2/8/2019,10:37,Ewallet,604.17,4.76,30.21,5.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,233-67-5758,C,Naypyitaw,Normal,Male,Health and beauty,40.35,1,2.02,42.37,1/29/2019,13:46,Ewallet,40.35,4.76,2.02,6.20
996,303-96-2227,B,Mandalay,Normal,Female,Home and lifestyle,97.38,10,48.69,"1,022.49",3/2/2019,17:16,Ewallet,973.80,4.76,48.69,4.40
997,727-02-1313,A,Yangon,Member,Male,Food and beverages,31.84,1,1.59,33.43,2/9/2019,13:22,Cash,31.84,4.76,1.59,7.70
998,347-56-2442,A,Yangon,Normal,Male,Home and lifestyle,65.82,1,3.29,69.11,2/22/2019,15:33,Cash,65.82,4.76,3.29,4.10


In [11]:

# ==========================================
# 1. METADATOS Y ESTADO ORIGINAL (ANTES)
# ==========================================
print("\n" + "="*50)
print(" METADATOS Y MUESTRA ORIGINAL (ANTES)")
print("="*50)

# Muestra una fracción real de los datos para entender el contexto
print("\n---> Muestra Aleatoria de 5 registros (df.sample):")
print(df.sample(5))

# Diagnostica la cardinalidad (valores únicos)
print("\n---> Metadatos: Valores Únicos Diferenciados por Columna:")
print(df.nunique())

# Diagnostica valores en cero que podrían ser errores de captura
print("\n---> Metadatos: Conteo exacto de Ceros (0) por Columna:")
print((df == 0).sum())

# Identifica nulos
print("\n---> Conteo de Valores Nulos:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if not nulos[nulos > 0].empty else "No se detectaron nulos.")

# Filtra el resumen estadístico a lo esencial
print("\n---> Resumen Estadístico Específico (Count, Min, Max):")
columnas_numericas = df.select_dtypes(include=[np.number]).columns
if len(columnas_numericas) > 0:
    resumen = df[columnas_numericas].describe().T
    print(resumen[['count', 'min', 'max']])



 METADATOS Y MUESTRA ORIGINAL (ANTES)

---> Muestra Aleatoria de 5 registros (df.sample):
      Invoice ID Branch      City Customer type  Gender  \
706  834-45-5519      B  Mandalay        Normal  Female   
626  816-57-2053      A    Yangon        Normal    Male   
762  410-67-1709      A    Yangon        Member  Female   
703  729-06-2010      B  Mandalay        Member    Male   
490  686-41-0932      B  Mandalay        Normal  Female   

               Product line  Unit price  Quantity  Tax 5%  Total       Date  \
706  Electronic accessories       43.00         4    8.60 180.60  1/31/2019   
626       Sports and travel       60.87         2    6.09 127.83   3/9/2019   
762     Fashion accessories       63.88         8   25.55 536.59  1/20/2019   
703       Health and beauty       80.47         9   36.21 760.44   1/6/2019   
490     Fashion accessories       34.70         2    3.47  72.87  3/13/2019   

      Time  Payment   cogs  gross margin percentage  gross income  Rating  
706

In [12]:

# ==========================================
# 2. PROCESAMIENTO Y LIMPIEZA DINÁMICA
# ==========================================
# Limpia los datos eliminando registros completamente duplicados
df = df.drop_duplicates()

# Imputa dinámicamente: Promedio para números, Moda para texto
for col in columnas_numericas:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())

columnas_categoricas = df.select_dtypes(include=['object']).columns
for col in columnas_categoricas:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])


In [13]:

# ==========================================
# 3. DIAGNÓSTICO DEL "DESPUÉS"
# ==========================================
print("\n" + "="*50)
print(" ESTADO LIMPIO DE LOS DATOS (DESPUÉS)")
print("="*50)

# Muestra los primeros registros para confirmar estructura final
print("\n---> Primeros 5 registros limpios (df.head):")
print(df.head(5))

# Confirma cambios en la estadística descriptiva
print("\n---> Nuevo Resumen Estadístico (Count, Min, Max):")
if len(columnas_numericas) > 0:
    resumen_limpio = df[columnas_numericas].describe().T
    print(resumen_limpio[['count', 'min', 'max']])

# Exporta el resultado final en UTF-8 para garantizar lectura en Looker
df.to_csv(ruta_salida, index=False, encoding='utf-8')

# Confirma el éxito de la operación
print(f"\nProceso finalizado. Archivo guardado en: {ruta_salida}")


 ESTADO LIMPIO DE LOS DATOS (DESPUÉS)

---> Primeros 5 registros limpios (df.head):
    Invoice ID Branch       City Customer type  Gender  \
0  750-67-8428      A     Yangon        Member  Female   
1  226-31-3081      C  Naypyitaw        Normal  Female   
2  631-41-3108      A     Yangon        Normal    Male   
3  123-19-1176      A     Yangon        Member    Male   
4  373-73-7910      A     Yangon        Normal    Male   

             Product line  Unit price  Quantity  Tax 5%  Total       Date  \
0       Health and beauty       74.69         7   26.14 548.97   1/5/2019   
1  Electronic accessories       15.28         5    3.82  80.22   3/8/2019   
2      Home and lifestyle       46.33         7   16.22 340.53   3/3/2019   
3       Health and beauty       58.22         8   23.29 489.05  1/27/2019   
4       Sports and travel       86.31         7   30.21 634.38   2/8/2019   

    Time      Payment   cogs  gross margin percentage  gross income  Rating  
0  13:08      Ewallet 522